# ML Models
Exécution des 3 modèles ML : BERTopic (topic modeling), NER (entités), Isolation Forest (anomalies).

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

data_path = project_root / 'data/processed/gdelt_benin_clean.csv'
df = pd.read_csv(data_path)
df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
print(f'Loaded {len(df)} rows from {data_path.name}')
print(f'Date range: {df["date_parsed"].min()} → {df["date_parsed"].max()}')

Loaded 5089 rows from gdelt_benin_clean.csv
Date range: 2025-01-01 00:00:00 → 2026-05-02 00:00:00


## Model 1: BERTopic (Topic Modeling)
Identifie les thèmes médiatiques récurrents autour du Bénin.

In [2]:
from src.models.topic_model import extract_topics

benin_mask = (
    df.get('Actor1CountryCode', pd.Series(dtype=str)).astype(str).eq('BEN') |
    df.get('ActionGeo_FullName', pd.Series(dtype=str)).astype(str).str.contains('benin', case=False, na=False)
 )
raw_texts = df.loc[benin_mask, 'event_label'].astype(str).dropna()
benin_texts = [t.strip() for t in raw_texts if isinstance(t, str) and t.strip() and len(t.strip()) > 10]
if len(benin_texts) == 0:
    print('No Benin-related event_label texts found — skipping BERTopic.')
    topic_model = None
    topic_info = pd.DataFrame(columns=['Topic', 'Count'])
else:
    sample_size = min(300, len(benin_texts))
    sample_texts = np.random.choice(benin_texts, sample_size, replace=False).tolist()
    sample_texts = [s for s in sample_texts if s and len(s.strip()) > 5]
    if len(sample_texts) < 20:
        print(f'Too few samples ({len(sample_texts)}) — skipping BERTopic.')
        topic_model = None
        topic_info = pd.DataFrame(columns=['Topic', 'Count'])
    else:
        print(f'Training BERTopic on {len(sample_texts)} samples...')
        result = extract_topics(sample_texts, min_topic_size=15)

if 'result' in locals():
    topic_model = result.model
    topics_list = result.topics
    topic_info = topic_model.get_topic_info()
    print(f'\nTopics found: {len(topic_info)}')
    print('\nTop 5 topics by frequency:')
    print(topic_info.head(5)[['Topic', 'Count']].to_string(index=False))
else:
    topic_model = None
    topics_list = []
    topic_info = pd.DataFrame(columns=['Topic', 'Count'])

Training BERTopic on 300 samples...


/home/appolinaire/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|                                                                            | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6208.98it/s]


Topics found: 8

Top 5 topics by frequency:
 Topic  Count
     0     84
     1     54
     2     37
     3     34
     4     31


In [3]:
print('\nTopic Keywords:')
for topic_id in topic_info['Topic'].head(5):
    if topic_id == -1:
        print(f'Topic {topic_id}: [Outliers]')
        continue
    keywords = topic_model.get_topic(topic_id)
    kw_str = ', '.join([kw for kw, _ in keywords[:8]])
    print(f'Topic {topic_id}: {kw_str}')


Topic Keywords:
Topic 0: consultation, , , , , , , 
Topic 1: publique, déclaration, protestation, , , , , 
Topic 2: diplomatique, engagement, , , , , , 
Topic 3: demande, appel, aide, assistance, , , , 
Topic 4: désapprobation, refus, rejet, , , , , 


## Model 2: NER (Named Entity Recognition)
Extrait les entités nommées : personnes, organisations, lieux.

In [4]:
from src.models.ner_model import extract_entities
from collections import Counter
import re

# Prepare text for NER
df['ner_text'] = (
    df['Actor1Name'].fillna('').astype(str) + ' ' +
    df['Actor2Name'].fillna('').astype(str) + ' ' +
    df['ActionGeo_FullName'].fillna('').astype(str)
).str.strip()

print('Extracting entities...')
entities = extract_entities(df['ner_text'].tolist())

# Aggregate
person_counter = Counter()
org_counter = Counter()
location_counter = Counter()

NOISE_TOKENS = {'INCONNU', 'UNKNOWN', 'NA', 'N/A', 'NONE', 'NULL'}
COUNTRY_NOISE = {'BENIN', 'BEN', 'BENINESE'}

def clean_entity(val, kind):
    text = re.sub(r'\s+', ' ', str(val or '')).strip()
    if not text:
        return ''
    parts = [p.strip() for p in re.split(r'[\s,;/|]+', text) if p.strip()]
    cleaned = []
    for p in parts:
        upper_p = p.upper()
        if upper_p in NOISE_TOKENS:
            continue
        if kind in ('persons', 'orgs') and upper_p in COUNTRY_NOISE:
            continue
        cleaned.append(p.title() if p.isupper() else p)
    result = ' '.join(cleaned).strip()
    return result if len(result) >= 3 and not result.isdigit() else ''

for entity_dict in entities:
    for v in entity_dict.get('persons', []):
        clean_v = clean_entity(v, 'persons')
        if clean_v:
            person_counter[clean_v] += 1
    for v in entity_dict.get('orgs', []):
        clean_v = clean_entity(v, 'orgs')
        if clean_v:
            org_counter[clean_v] += 1
    for v in entity_dict.get('locations', []):
        clean_v = clean_entity(v, 'locations')
        if clean_v:
            location_counter[clean_v] += 1

print(f'\nExtracted {len(person_counter)} persons, {len(org_counter)} orgs, {len(location_counter)} locations')
print('\nTop 10 Persons:')
for person, count in person_counter.most_common(10):
    print(f'  {person}: {count}')
print('\nTop 10 Organizations:')
for org, count in org_counter.most_common(10):
    print(f'  {org}: {count}')
print('\nTop 10 Locations:')
for loc, count in location_counter.most_common(10):
    print(f'  {loc}: {count}')

Extracting entities...



Extracted 237 persons, 608 orgs, 31 locations

Top 10 Persons:
  Atlantique: 121
  Zou: 91
  Qué: 48
  Cotonou: 37
  Porto-Novo: 33
  Ouidah: 31
  Abomey: 30
  Africa: 24
  Alibori: 21
  Kandi: 20

Top 10 Organizations:
  Alibori: 148
  Africa: 89
  Niger: 66
  Nigeria: 36
  Ouidah: 29
  Minist: 27
  British: 26
  United Kingdom: 25
  Community: 24
  Business: 20

Top 10 Locations:
  Benin: 218
  Borgou: 55
  France: 32
  Egypt: 23
  Donga: 20
  Atakora: 13
  West African States: 10
  Parakou: 8
  Kouffo: 8
  Haiti: 7


## Model 3: Isolation Forest (Anomaly Detection)
Détecte les mois avec patterns anormaux (pics de volume, sentiment ou activité géopolitique).

In [5]:
from src.models.anomaly_model import detect_anomalies
from src.models.baseline_anomaly import detect_anomalies_iqr

ml_monthly = (
    df.assign(month=df['date_parsed'].dt.to_period('M'))
      .groupby('month', as_index=False)
      .agg(
          rows=('event_label', 'size'),
          avg_tone=('AvgTone', 'mean'),
          goldstein_scale=('GoldsteinScale', 'mean'),
          num_articles=('NumArticles', 'sum'),
      )
      .dropna()
      .sort_values('month')
      .reset_index(drop=True)
)

print(f'Monthly aggregated data: {len(ml_monthly)} months')
print('\nMonthly stats:')
print(ml_monthly[['month', 'rows', 'avg_tone', 'goldstein_scale']].head(10).to_string(index=False))

Monthly aggregated data: 15 months

Monthly stats:
  month  rows  avg_tone  goldstein_scale
2025-01   705 -1.106617         0.580426
2025-02   293 -1.380411         0.869625
2025-03   322 -1.162040         0.808075
2025-04   171 -0.121355         1.680117
2025-05   219 -0.570265         1.433333
2025-06    75 -0.419531         1.332000
2025-07    25 -0.575261         1.392000
2025-10    57 -0.656073         1.138596
2025-11    73 -0.881445        -0.041096
2025-12   173 -2.773172        -0.758382


In [6]:
# Run Isolation Forest
print('Running Isolation Forest...')
anomaly_result = detect_anomalies(
    dataframe=ml_monthly,
    feature_columns=['rows', 'avg_tone', 'goldstein_scale', 'num_articles'],
    contamination=0.1,
    random_state=42,
)

# Run IQR baseline
baseline_result = detect_anomalies_iqr(
    dataframe=ml_monthly,
    feature_columns=['rows', 'avg_tone', 'goldstein_scale', 'num_articles'],
)

if_anomalies = anomaly_result.dataframe[anomaly_result.dataframe['is_anomaly']]
iqr_anomalies = baseline_result.dataframe[baseline_result.dataframe['iqr_is_anomaly']]

print(f'\nIsolation Forest detected {len(if_anomalies)} anomalous months:')
if len(if_anomalies) > 0:
    print(if_anomalies[['month', 'rows', 'anomaly_score']].sort_values('anomaly_score', ascending=False).to_string(index=False))

print(f'\nIQR baseline detected {len(iqr_anomalies)} anomalous months:')
if len(iqr_anomalies) > 0:
    print(iqr_anomalies[['month', 'rows', 'iqr_outlier_count']].to_string(index=False))

Running Isolation Forest...



Isolation Forest detected 2 anomalous months:
  month  rows  anomaly_score
2025-12   173       0.612221
2026-05     8       0.559576

IQR baseline detected 0 anomalous months:


In [7]:
print('\n=== Comparative Summary ==="')
comparison = anomaly_result.dataframe[['month', 'rows', 'avg_tone', 'goldstein_scale', 'num_articles', 'anomaly_score', 'is_anomaly']].copy()
comparison['iqr_is_anomaly'] = baseline_result.dataframe['iqr_is_anomaly'].values
comparison['iqr_outlier_count'] = baseline_result.dataframe['iqr_outlier_count'].values
print(comparison.sort_values('anomaly_score', ascending=False).head(15).to_string(index=False))


=== Comparative Summary ==="
  month  rows  avg_tone  goldstein_scale  num_articles  anomaly_score  is_anomaly  iqr_is_anomaly  iqr_outlier_count
2025-12   173 -2.773172        -0.758382          1364       0.612221        True           False                  0
2026-05     8 -2.541334         0.187500            44       0.559576        True           False                  0
2026-02   791 -2.063033         0.147661          5200       0.544913       False           False                  0
2025-04   171 -0.121355         1.680117          1186       0.542822       False           False                  0
2026-01   817 -1.587441         0.789718          5654       0.534887       False           False                  0
2026-04   640 -1.758595         0.505312          4097       0.505661       False           False                  0
2025-11    73 -0.881445        -0.041096           581       0.498612       False           False                  0
2025-01   705 -1.106617         0.

## Summary
- **BERTopic**: Extracts semantic topics from event labels (e.g., security, diplomacy, governance).
- **NER**: Identifies key persons, organizations, and locations from GDELT structured fields.
- **Isolation Forest**: Flags months with anomalous event patterns (volume, sentiment, conflict intensity).

## Model 4: Media Impact Predictor (Saved Model)
Charge le modele sauvegarde et verifie une prediction rapide sur un echantillon.

In [8]:
from pathlib import Path
import sys
import joblib
from src.models.media_model import load_model, train_and_save_media_model

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

model_path = project_root / "models" / "media_model" / "media_rf.pkl"
enc_path = project_root / "models" / "media_model" / "media_encoders.joblib"

if not model_path.exists() or not enc_path.exists():
    if "NumArticles" in df.columns:
        df["is_high_media"] = (df["NumArticles"] > df["NumArticles"].median()).astype(int)
    else:
        df["is_high_media"] = 0

    if "Actor2CountryCode" in df.columns:
        df["is_international"] = (df["Actor2CountryCode"] != "BEN").astype(int)
    else:
        df["is_international"] = 0

    features = ["GoldsteinScale", "EventRootCode", "is_international"]
    output_dir = project_root / "models" / "media_model"

    artifacts = train_and_save_media_model(
        df=df,
        features=features,
        target="is_high_media",
        output_dir=output_dir,
    )
    print("Saved model:", artifacts.model_path)
    print("Saved encoders:", artifacts.encoder_path)

model = load_model(model_path)
encoders = joblib.load(enc_path)

features = ["GoldsteinScale", "EventRootCode", "is_international"]

sample_df = df.copy()
if "is_international" not in sample_df.columns:
    sample_df["is_international"] = (sample_df["Actor2CountryCode"] != "BEN").astype(int)

X = sample_df[features].dropna().head(10).copy()

if "EventRootCode" in encoders:
    X["EventRootCode"] = encoders["EventRootCode"].transform(X["EventRootCode"].astype(str))

preds = model.predict(X)
print("Sample predictions:", preds)

              precision    recall  f1-score   support

           0       0.55      0.63      0.59       503
           1       0.58      0.49      0.53       515

    accuracy                           0.56      1018
   macro avg       0.56      0.56      0.56      1018
weighted avg       0.56      0.56      0.56      1018

Saved model: /home/appolinaire/projects/benin-insights/models/media_model/media_rf.pkl
Saved encoders: /home/appolinaire/projects/benin-insights/models/media_model/media_encoders.joblib
Sample predictions: [0 1 1 1 0 1 1 0 1 0]


## Model 4: Media Impact Predictor (Train + Save)
Entraine le modele, sauvegarde dans `models/media_model/`, puis affiche les metriques de base.

In [9]:
from pathlib import Path
import sys
from src.models.media_model import train_and_save_media_model

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

# Prepare columns used by the media model
if "NumArticles" in df.columns:
    df["is_high_media"] = (df["NumArticles"] > df["NumArticles"].median()).astype(int)
else:
    df["is_high_media"] = 0

if "Actor2CountryCode" in df.columns:
    df["is_international"] = (df["Actor2CountryCode"] != "BEN").astype(int)
else:
    df["is_international"] = 0

features = ["GoldsteinScale", "EventRootCode", "is_international"]
output_dir = project_root / "models" / "media_model"

artifacts = train_and_save_media_model(
    df=df,
    features=features,
    target="is_high_media",
    output_dir=output_dir,
)
print("Saved model:", artifacts.model_path)
print("Saved encoders:", artifacts.encoder_path)

              precision    recall  f1-score   support

           0       0.55      0.63      0.59       503
           1       0.58      0.49      0.53       515

    accuracy                           0.56      1018
   macro avg       0.56      0.56      0.56      1018
weighted avg       0.56      0.56      0.56      1018



Saved model: /home/appolinaire/projects/benin-insights/models/media_model/media_rf.pkl
Saved encoders: /home/appolinaire/projects/benin-insights/models/media_model/media_encoders.joblib


In [10]:
# Diagnostics for low accuracy
class_counts = df["is_high_media"].value_counts(dropna=False)
class_ratio = (class_counts / class_counts.sum()).round(3)
print("Class distribution (is_high_media):")
print(class_ratio)

baseline_accuracy = class_ratio.max()
print("Baseline accuracy (majority class):", baseline_accuracy)

print("Missing values in features:")
print(df[features].isna().mean().round(3))

print("Unique EventRootCode values:", df["EventRootCode"].nunique())
print("Top EventRootCode counts:")
print(df["EventRootCode"].value_counts().head(10).to_string())

Class distribution (is_high_media):
is_high_media
0    0.514
1    0.486
Name: count, dtype: float64
Baseline accuracy (majority class): 0.514
Missing values in features:
GoldsteinScale      0.0
EventRootCode       0.0
is_international    0.0
dtype: float64
Unique EventRootCode values: 19
Top EventRootCode counts:
EventRootCode
4     1358
1      713
5      611
2      340
11     325
19     297
17     218
8      187
3      178
7      175
